# Microproyecto 2 - Clasificacion de textos ODS (Experimentacion exhaustiva)

Este notebook implementa todo el plan de experimentacion:

- Preprocesamiento textual configurable: tokenizacion, normalizacion, stopwords, lematizacion y stemming.
- Representaciones: TF-IDF y embeddings (clasicos + Hugging Face).
- Reduccion de dimensionalidad: SVD, NMF y PCA.
- Modelos: Naive Bayes, CatBoost, LightGBM.
- Protocolo: split `train/val/test`, tuning sobre `train`, seleccion por `val`, evaluacion y ranking final en `test`.

El notebook selecciona el ganador y exporta artefactos en `artifacts/`.


In [1]:
import os
import re
import json
import time
import random
import warnings
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('Seed:', SEED)

Seed: 42


In [2]:
# Configuracion del experimento
USE_DEV_SAMPLE = os.getenv('USE_DEV_SAMPLE', '0') == '1'
DEV_SAMPLE_SIZE = int(os.getenv('DEV_SAMPLE_SIZE', '2400'))
FAST_MODE = os.getenv('FAST_MODE', '0') == '1'
MAX_EXPERIMENTS = int(os.getenv('MAX_EXPERIMENTS', '0'))  # 0 => todos

ENABLE_HF_EXPERIMENTS = os.getenv('ENABLE_HF_EXPERIMENTS', '1') == '1'
REQUIRE_MPS_FOR_HF = os.getenv('REQUIRE_MPS_FOR_HF', '1') == '1'
HF_MODEL_NAME = os.getenv('HF_MODEL_NAME', 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
HF_N_ITER = int(os.getenv('HF_N_ITER', '2'))

# Parametro de validacion rapida
LIMIT_PREPROC_VARIANTS = os.getenv('LIMIT_PREPROC_VARIANTS', '0') == '1'

print('USE_DEV_SAMPLE:', USE_DEV_SAMPLE)
print('DEV_SAMPLE_SIZE:', DEV_SAMPLE_SIZE)
print('FAST_MODE:', FAST_MODE)
print('MAX_EXPERIMENTS:', MAX_EXPERIMENTS)
print('ENABLE_HF_EXPERIMENTS:', ENABLE_HF_EXPERIMENTS)
print('REQUIRE_MPS_FOR_HF:', REQUIRE_MPS_FOR_HF)
print('HF_MODEL_NAME:', HF_MODEL_NAME)
print('HF_N_ITER:', HF_N_ITER)
print('LIMIT_PREPROC_VARIANTS:', LIMIT_PREPROC_VARIANTS)

USE_DEV_SAMPLE: False
DEV_SAMPLE_SIZE: 2400
FAST_MODE: False
MAX_EXPERIMENTS: 0
ENABLE_HF_EXPERIMENTS: True
REQUIRE_MPS_FOR_HF: True
HF_MODEL_NAME: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
HF_N_ITER: 2
LIMIT_PREPROC_VARIANTS: False


In [3]:
# Import de librerias ML/NLP
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import train_test_split, ParameterGrid, ParameterSampler
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, NMF, PCA
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.naive_bayes import ComplementNB, GaussianNB

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

import nltk
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
import spacy

from gensim.models import FastText, Word2Vec
from sentence_transformers import SentenceTransformer

import torch
from IPython.display import display

print('Imports cargados correctamente.')

Imports cargados correctamente.


In [4]:
# Validacion de dispositivo para embeddings HF (PyTorch + MPS)
mps_available = torch.backends.mps.is_available()
mps_built = torch.backends.mps.is_built()

print('torch version:', torch.__version__)
print('MPS built    :', mps_built)
print('MPS available:', mps_available)

if ENABLE_HF_EXPERIMENTS:
    if REQUIRE_MPS_FOR_HF and not mps_available:
        raise RuntimeError(
            'HF embeddings requieren GPU Mac via MPS. '
            'En este entorno MPS no esta disponible. '
            'Si necesitas correr localmente sin HF, define ENABLE_HF_EXPERIMENTS=0.'
        )
    HF_DEVICE = 'mps' if mps_available else 'cpu'
else:
    HF_DEVICE = 'cpu'

print('HF_DEVICE:', HF_DEVICE)

torch version: 2.10.0
MPS built    : True
MPS available: True
HF_DEVICE: mps


In [5]:
# Recursos linguisticos
NLTK_DATA_DIR = Path('nltk_data')
NLTK_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Fuerza a NLTK a usar directorio local del proyecto (evita errores de permisos en HOME)
if str(NLTK_DATA_DIR.resolve()) not in nltk.data.path:
    nltk.data.path.insert(0, str(NLTK_DATA_DIR.resolve()))

nltk.download('stopwords', download_dir=str(NLTK_DATA_DIR.resolve()), quiet=True)
SPANISH_STOPWORDS = set(stopwords.words('spanish'))

_SPACY_MODEL = None

def get_spacy_model():
    global _SPACY_MODEL
    if _SPACY_MODEL is None:
        try:
            _SPACY_MODEL = spacy.load('es_core_news_sm', disable=['parser', 'ner'])
        except OSError:
            from spacy.cli import download
            download('es_core_news_sm')
            _SPACY_MODEL = spacy.load('es_core_news_sm', disable=['parser', 'ner'])
    return _SPACY_MODEL

print('Stopwords ES:', len(SPANISH_STOPWORDS))
print('NLTK_DATA_DIR:', NLTK_DATA_DIR.resolve())

Stopwords ES: 313
NLTK_DATA_DIR: /Users/adrianalarcon/Library/CloudStorage/OneDrive-UniversidaddelosAndes/MAIA/aprendizaje_no_supervisado/assignment_2/nltk_data


In [6]:
# Carga de datos
DATA_PATH = Path('data/train.xlsx')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'No existe {DATA_PATH}')

df = pd.read_excel(DATA_PATH)
df = df.dropna(subset=['textos', 'ODS']).copy()
df['textos'] = df['textos'].astype(str)
df['ODS'] = df['ODS'].astype(int)

if USE_DEV_SAMPLE and DEV_SAMPLE_SIZE < len(df):
    df, _ = train_test_split(
        df,
        train_size=DEV_SAMPLE_SIZE,
        stratify=df['ODS'],
        random_state=SEED,
    )
    df = df.reset_index(drop=True)

print('Shape final:', df.shape)
print('ODS unicos:', sorted(df['ODS'].unique().tolist()))
print('Distribucion clases:')
print(df['ODS'].value_counts().sort_index())

df.head(3)

Shape final: (9656, 2)
ODS unicos: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
Distribucion clases:
ODS
1      505
2      369
3      894
4     1025
5     1070
6      695
7      787
8      446
9      343
10     352
11     607
12     312
13     464
14     377
15     330
16    1080
Name: count, dtype: int64


,textos,ODS
0,"""Aprendizaje"" y ""educación"" se consideran sinó...",4
1,No dejar clara la naturaleza de estos riesgos ...,6
2,"Como resultado, un mayor y mejorado acceso al ...",13


In [7]:
# Split train / val / test
X = df['textos']
y = df['ODS']

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.15,
    stratify=y,
    random_state=SEED,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.17647,  # para 70/15/15 aprox
    stratify=y_train_val,
    random_state=SEED,
)

print('train:', len(X_train))
print('val  :', len(X_val))
print('test :', len(X_test))

train: 6758
val  : 1449
test : 1449


In [ ]:
# Transformadores personalizados (autocontenidos en notebook)
import hashlib


class TextPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        lowercase=True,
        strip_accents=True,
        remove_stopwords=False,
        lemmatize=False,
        stem=False,
        min_token_len=2,
        stopword_language='spanish',
        spacy_model_name='es_core_news_sm',
        nltk_data_dir='nltk_data',
    ):
        self.lowercase = lowercase
        self.strip_accents = strip_accents
        self.remove_stopwords = remove_stopwords
        self.lemmatize = lemmatize
        self.stem = stem
        self.min_token_len = min_token_len
        self.stopword_language = stopword_language
        self.spacy_model_name = spacy_model_name
        self.nltk_data_dir = nltk_data_dir

    def fit(self, X, y=None):
        data_dir = Path(self.nltk_data_dir).resolve()
        data_dir.mkdir(parents=True, exist_ok=True)
        if str(data_dir) not in nltk.data.path:
            nltk.data.path.insert(0, str(data_dir))

        nltk.download('stopwords', download_dir=str(data_dir), quiet=True)
        self._stopwords = set(stopwords.words(self.stopword_language))
        self._stemmer = SnowballStemmer(self.stopword_language)

        if self.lemmatize:
            try:
                self._nlp = spacy.load(self.spacy_model_name, disable=['parser', 'ner'])
            except OSError:
                from spacy.cli import download
                download(self.spacy_model_name)
                self._nlp = spacy.load(self.spacy_model_name, disable=['parser', 'ner'])
        else:
            self._nlp = None

        return self

    def _normalize(self, text):
        text = str(text)
        if self.lowercase:
            text = text.lower()
        if self.strip_accents:
            text = ''.join(
                c for c in unicodedata.normalize('NFKD', text)
                if not unicodedata.combining(c)
            )
        return text

    def _tokenize(self, text):
        return re.findall(r'[a-zA-ZñÑüÜ0-9]+', text)

    def transform(self, X):
        processed = []
        for text in X:
            text = self._normalize(text)
            toks = self._tokenize(text)
            toks = [t for t in toks if len(t) >= self.min_token_len]

            if self.remove_stopwords:
                toks = [t for t in toks if t not in self._stopwords]

            if self.lemmatize and self._nlp is not None:
                doc = self._nlp(' '.join(toks))
                toks = [tok.lemma_.strip() for tok in doc if tok.lemma_.strip()]

            if self.stem:
                toks = [self._stemmer.stem(t) for t in toks]

            processed.append(' '.join(toks))

        return processed


class MeanGensimEmbeddingTransformer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        method='word2vec',
        vector_size=150,
        window=5,
        min_count=2,
        epochs=20,
        workers=1,
        seed=42,
    ):
        self.method = method
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.epochs = epochs
        self.workers = workers
        self.seed = seed

    def fit(self, X, y=None):
        tokenized = [str(x).split() for x in X]
        if self.method == 'fasttext':
            self.model_ = FastText(
                sentences=tokenized,
                vector_size=self.vector_size,
                window=self.window,
                min_count=self.min_count,
                epochs=self.epochs,
                workers=self.workers,
                seed=self.seed,
            )
        else:
            self.model_ = Word2Vec(
                sentences=tokenized,
                vector_size=self.vector_size,
                window=self.window,
                min_count=self.min_count,
                epochs=self.epochs,
                workers=self.workers,
                seed=self.seed,
            )
        return self

    def transform(self, X):
        vectors = np.zeros((len(X), self.vector_size), dtype=np.float32)
        for i, text in enumerate(X):
            toks = str(text).split()
            vecs = [self.model_.wv[t] for t in toks if t in self.model_.wv]
            if vecs:
                vectors[i] = np.mean(vecs, axis=0)
        return vectors


class HFEmbeddingTransformer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
        device='cpu',
        batch_size=32,
        normalize_embeddings=True,
        use_cache=True,
        cache_dir='cache/hf_embeddings',
    ):
        self.model_name = model_name
        self.device = device
        self.batch_size = batch_size
        self.normalize_embeddings = normalize_embeddings
        self.use_cache = use_cache
        self.cache_dir = cache_dir

    def fit(self, X, y=None):
        self.model_ = SentenceTransformer(self.model_name, device=self.device)
        return self

    def _build_cache_path(self, X):
        hasher = hashlib.sha1()
        hasher.update(self.model_name.encode('utf-8'))
        hasher.update(str(self.normalize_embeddings).encode('utf-8'))
        hasher.update(str(len(X)).encode('utf-8'))
        for text in X:
            s = str(text)
            hasher.update(s.encode('utf-8', errors='ignore'))
            hasher.update(b'\x00')
        return Path(self.cache_dir) / f"{hasher.hexdigest()}.npy"

    def transform(self, X):
        texts = list(map(str, X))

        if self.use_cache:
            cache_path = self._build_cache_path(texts)
            if cache_path.exists():
                return np.load(cache_path).astype(np.float32)

        if not hasattr(self, 'model_'):
            self.model_ = SentenceTransformer(self.model_name, device=self.device)

        emb = self.model_.encode(
            texts,
            batch_size=self.batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=self.normalize_embeddings,
        )
        emb = emb.astype(np.float32)

        if self.use_cache:
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            np.save(cache_path, emb)

        return emb




In [9]:
# Preview de preprocessing
preview = pd.DataFrame({'raw': X_train.reset_index(drop=True).head(3)})

preview['basic'] = TextPreprocessor(remove_stopwords=False, lemmatize=False, stem=False).fit_transform(preview['raw'])
preview['stop'] = TextPreprocessor(remove_stopwords=True, lemmatize=False, stem=False).fit_transform(preview['raw'])
preview['lemma_stop'] = TextPreprocessor(remove_stopwords=True, lemmatize=True, stem=False).fit_transform(preview['raw'])
preview['stem_stop'] = TextPreprocessor(remove_stopwords=True, lemmatize=False, stem=True).fit_transform(preview['raw'])

preview

,raw,basic,stop,lemma_stop,stem_stop
0,"Tirpak et al (2010) recomiendan que, dada la n...",tirpak et al 2010 recomiendan que dada la natu...,tirpak et 2010 recomiendan dada naturaleza con...,tirpak et 2010 recomendar dado naturaleza cont...,tirpak et 2010 recomiend dad naturalez context...
1,Un liderazgo equilibrado y una formulación de ...,un liderazgo equilibrado una formulacion de po...,liderazgo equilibrado formulacion politicas se...,liderazgo equilibrado formulacion politica sen...,liderazg equilibr formul polit sensibl gener m...
2,Si bien la generalización de las fallas graves...,si bien la generalizacion de las fallas graves...,si bien generalizacion fallas graves mercado s...,si bien generalizacion falla grave mercado sec...,si bien generaliz fall grav merc sector atenci...


In [10]:
# Utilidades de experimentacion (sin CV)

# Variantes linguisticas obligatorias
PREPROC_VARIANTS = [
    {'prep__remove_stopwords': [False], 'prep__lemmatize': [False], 'prep__stem': [False]},
    {'prep__remove_stopwords': [True],  'prep__lemmatize': [False], 'prep__stem': [False]},
    {'prep__remove_stopwords': [True],  'prep__lemmatize': [True],  'prep__stem': [False]},
    {'prep__remove_stopwords': [True],  'prep__lemmatize': [False], 'prep__stem': [True]},
]

if LIMIT_PREPROC_VARIANTS:
    PREPROC_VARIANTS = PREPROC_VARIANTS[:2]


def add_preproc_variants(base):
    spaces = []
    for variant in PREPROC_VARIANTS:
        p = dict(base)
        p.update(variant)
        spaces.append(p)
    return spaces


def build_param_candidates(param_space, search_kind, n_iter, seed):
    spaces = param_space if isinstance(param_space, list) else [param_space]

    if search_kind == 'grid':
        candidates = []
        for sp in spaces:
            candidates.extend(list(ParameterGrid(sp)))
    else:
        candidates = []
        n_spaces = len(spaces)
        base = max(1, n_iter // n_spaces)
        rem = max(0, n_iter - base * n_spaces)
        for i, sp in enumerate(spaces):
            n_local = base + (1 if i < rem else 0)
            n_local = max(1, n_local)
            sampled = list(ParameterSampler(sp, n_iter=n_local, random_state=seed + i))
            candidates.extend(sampled)

    # deduplicacion
    unique = []
    seen = set()
    for p in candidates:
        key = tuple(sorted((k, repr(v)) for k, v in p.items()))
        if key not in seen:
            seen.add(key)
            unique.append(p)

    return unique


def run_experiment(exp_cfg):
    name = exp_cfg['name']
    search_kind = exp_cfg.get('search_kind', 'random')
    n_iter = exp_cfg.get('n_iter', 6 if not FAST_MODE else 3)

    print(f"\n===== {name} =====")
    t0 = time.perf_counter()

    candidates = build_param_candidates(exp_cfg['params'], search_kind, n_iter, SEED)
    if len(candidates) == 0:
        raise RuntimeError(f'No se generaron candidatos para {name}')

    best_val = -1.0
    best_model = None
    best_params = None

    for params in candidates:
        model = clone(exp_cfg['pipeline'])
        model.set_params(**params)
        model.fit(X_train, y_train)

        y_val_pred = model.predict(X_val)
        val_f1 = float(f1_score(y_val, y_val_pred, average='macro'))

        if val_f1 > best_val:
            best_val = val_f1
            best_model = model
            best_params = params

    y_val_pred = best_model.predict(X_val)
    y_test_pred = best_model.predict(X_test)

    elapsed = time.perf_counter() - t0

    row = {
        'experiment': name,
        'status': 'ok',
        'val_f1_macro': float(f1_score(y_val, y_val_pred, average='macro')),
        'val_f1_weighted': float(f1_score(y_val, y_val_pred, average='weighted')),
        'val_accuracy': float(accuracy_score(y_val, y_val_pred)),
        'test_f1_macro': float(f1_score(y_test, y_test_pred, average='macro')),
        'test_f1_weighted': float(f1_score(y_test, y_test_pred, average='weighted')),
        'test_accuracy': float(accuracy_score(y_test, y_test_pred)),
        'fit_time_sec': round(elapsed, 2),
        'n_candidates': len(candidates),
        'best_params': best_params,
    }

    print('VAL f1_macro :', round(row['val_f1_macro'], 4))
    print('TEST f1_macro:', round(row['test_f1_macro'], 4))
    print('Candidatos   :', row['n_candidates'])
    print('Tiempo (s)   :', row['fit_time_sec'])

    result = {
        'best_model': best_model,
        'best_params': best_params,
        'n_candidates': len(candidates),
    }

    return row, result

In [11]:
# Matriz de experimentos
num_classes = y_train.nunique()
cb_iters = [200, 350] if not FAST_MODE else [120, 200]
lgb_estimators = [250, 450] if not FAST_MODE else [150, 250]
rand_iter = 8 if not FAST_MODE else 3

experiments = [
    # --- Rama TF-IDF ---
    {
        'name': 'E01_Preproc_TFIDF_ComplementNB',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('tfidf', TfidfVectorizer(max_features=50000)),
            ('clf', ComplementNB()),
        ]),
        'params': add_preproc_variants({
            'tfidf__ngram_range': [(1, 1), (1, 2)],
            'tfidf__min_df': [2, 5],
            'tfidf__sublinear_tf': [True, False],
            'clf__alpha': [0.3, 0.7, 1.0],
        }),
    },
    {
        'name': 'E02_Preproc_TFIDF_SVD_CatBoost',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('tfidf', TfidfVectorizer(max_features=50000)),
            ('svd', TruncatedSVD(random_state=SEED)),
            ('clf', CatBoostClassifier(
                loss_function='MultiClass',
                random_state=SEED,
                verbose=0,
                allow_writing_files=False,
                thread_count=-1,
            )),
        ]),
        'params': add_preproc_variants({
            'tfidf__ngram_range': [(1, 1), (1, 2)],
            'tfidf__min_df': [2, 5],
            'svd__n_components': [120, 220, 320],
            'clf__depth': [6, 8],
            'clf__learning_rate': [0.03, 0.05, 0.1],
            'clf__iterations': cb_iters,
        }),
    },
    {
        'name': 'E03_Preproc_TFIDF_SVD_LightGBM',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('tfidf', TfidfVectorizer(max_features=50000)),
            ('svd', TruncatedSVD(random_state=SEED)),
            ('clf', LGBMClassifier(
                objective='multiclass',
                num_class=num_classes,
                random_state=SEED,
                n_jobs=-1,
                verbosity=-1,
            )),
        ]),
        'params': add_preproc_variants({
            'tfidf__ngram_range': [(1, 1), (1, 2)],
            'tfidf__min_df': [2, 5],
            'svd__n_components': [120, 220, 320],
            'clf__num_leaves': [31, 63],
            'clf__learning_rate': [0.03, 0.05, 0.1],
            'clf__n_estimators': lgb_estimators,
        }),
    },
    {
        'name': 'E04_Preproc_TFIDF_NMF_LightGBM',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('tfidf', TfidfVectorizer(max_features=50000)),
            ('nmf', NMF(random_state=SEED, init='nndsvda', max_iter=500)),
            ('clf', LGBMClassifier(
                objective='multiclass',
                num_class=num_classes,
                random_state=SEED,
                n_jobs=-1,
                verbosity=-1,
            )),
        ]),
        'params': add_preproc_variants({
            'tfidf__ngram_range': [(1, 1), (1, 2)],
            'tfidf__min_df': [2, 5],
            'nmf__n_components': [80, 140, 200],
            'clf__num_leaves': [31, 63],
            'clf__learning_rate': [0.03, 0.05, 0.1],
            'clf__n_estimators': lgb_estimators,
        }),
    },

    # --- Embeddings clasicos + Naive Bayes ---
    {
        'name': 'E05_Preproc_Word2Vec_GaussianNB',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('emb', MeanGensimEmbeddingTransformer(method='word2vec', seed=SEED, workers=1)),
            ('clf', GaussianNB()),
        ]),
        'params': add_preproc_variants({
            'emb__vector_size': [100, 150, 200],
            'emb__window': [5, 8],
            'emb__min_count': [1, 2],
            'emb__epochs': [20, 30],
            'clf__var_smoothing': [1e-9, 1e-8, 1e-7],
        }),
    },
    {
        'name': 'E06_Preproc_Word2Vec_PCA_GaussianNB',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('emb', MeanGensimEmbeddingTransformer(method='word2vec', vector_size=200, seed=SEED, workers=1)),
            ('pca', PCA(random_state=SEED)),
            ('clf', GaussianNB()),
        ]),
        'params': add_preproc_variants({
            'emb__window': [5, 8],
            'emb__min_count': [1, 2],
            'emb__epochs': [20, 30],
            'pca__n_components': [40, 80, 120],
            'clf__var_smoothing': [1e-9, 1e-8, 1e-7],
        }),
    },
    {
        'name': 'E07_Preproc_FastText_GaussianNB',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('emb', MeanGensimEmbeddingTransformer(method='fasttext', seed=SEED, workers=1)),
            ('clf', GaussianNB()),
        ]),
        'params': add_preproc_variants({
            'emb__vector_size': [100, 150, 200],
            'emb__window': [5, 8],
            'emb__min_count': [1, 2],
            'emb__epochs': [20, 30],
            'clf__var_smoothing': [1e-9, 1e-8, 1e-7],
        }),
    },
    {
        'name': 'E08_Preproc_FastText_PCA_GaussianNB',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('emb', MeanGensimEmbeddingTransformer(method='fasttext', vector_size=200, seed=SEED, workers=1)),
            ('pca', PCA(random_state=SEED)),
            ('clf', GaussianNB()),
        ]),
        'params': add_preproc_variants({
            'emb__window': [5, 8],
            'emb__min_count': [1, 2],
            'emb__epochs': [20, 30],
            'pca__n_components': [40, 80, 120],
            'clf__var_smoothing': [1e-9, 1e-8, 1e-7],
        }),
    },

    # --- Embeddings clasicos + CatBoost/LightGBM ---
    {
        'name': 'E09_Preproc_Word2Vec_CatBoost',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('emb', MeanGensimEmbeddingTransformer(method='word2vec', seed=SEED, workers=1)),
            ('clf', CatBoostClassifier(
                loss_function='MultiClass',
                random_state=SEED,
                verbose=0,
                allow_writing_files=False,
                thread_count=-1,
            )),
        ]),
        'params': add_preproc_variants({
            'emb__vector_size': [100, 150, 200],
            'emb__window': [5, 8],
            'emb__min_count': [1, 2],
            'emb__epochs': [20, 30],
            'clf__depth': [6, 8],
            'clf__learning_rate': [0.03, 0.05, 0.1],
            'clf__iterations': cb_iters,
        }),
    },
    {
        'name': 'E10_Preproc_Word2Vec_PCA_LightGBM',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('emb', MeanGensimEmbeddingTransformer(method='word2vec', vector_size=200, seed=SEED, workers=1)),
            ('pca', PCA(random_state=SEED)),
            ('clf', LGBMClassifier(
                objective='multiclass',
                num_class=num_classes,
                random_state=SEED,
                n_jobs=-1,
                verbosity=-1,
            )),
        ]),
        'params': add_preproc_variants({
            'emb__window': [5, 8],
            'emb__min_count': [1, 2],
            'emb__epochs': [20, 30],
            'pca__n_components': [40, 80, 120],
            'clf__num_leaves': [31, 63],
            'clf__learning_rate': [0.03, 0.05, 0.1],
            'clf__n_estimators': lgb_estimators,
        }),
    },
    {
        'name': 'E11_Preproc_FastText_LightGBM',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('emb', MeanGensimEmbeddingTransformer(method='fasttext', seed=SEED, workers=1)),
            ('clf', LGBMClassifier(
                objective='multiclass',
                num_class=num_classes,
                random_state=SEED,
                n_jobs=-1,
                verbosity=-1,
            )),
        ]),
        'params': add_preproc_variants({
            'emb__vector_size': [100, 150, 200],
            'emb__window': [5, 8],
            'emb__min_count': [1, 2],
            'emb__epochs': [20, 30],
            'clf__num_leaves': [31, 63],
            'clf__learning_rate': [0.03, 0.05, 0.1],
            'clf__n_estimators': lgb_estimators,
        }),
    },
    {
        'name': 'E12_Preproc_FastText_PCA_CatBoost',
        'search_kind': 'random',
        'n_iter': rand_iter,
        'pipeline': Pipeline([
            ('prep', TextPreprocessor()),
            ('emb', MeanGensimEmbeddingTransformer(method='fasttext', vector_size=200, seed=SEED, workers=1)),
            ('pca', PCA(random_state=SEED)),
            ('clf', CatBoostClassifier(
                loss_function='MultiClass',
                random_state=SEED,
                verbose=0,
                allow_writing_files=False,
                thread_count=-1,
            )),
        ]),
        'params': add_preproc_variants({
            'emb__window': [5, 8],
            'emb__min_count': [1, 2],
            'emb__epochs': [20, 30],
            'pca__n_components': [40, 80, 120],
            'clf__depth': [6, 8],
            'clf__learning_rate': [0.03, 0.05, 0.1],
            'clf__iterations': cb_iters,
        }),
    },
]

if ENABLE_HF_EXPERIMENTS:
    hf_experiments = [
        # Solo 2 experimentos avanzados: sin PCA y con PCA
        {
            'name': 'E13_Preproc_HF_GaussianNB',
            'search_kind': 'random',
            'n_iter': HF_N_ITER,
            'pipeline': Pipeline([
                ('prep', TextPreprocessor()),
                ('emb', HFEmbeddingTransformer(
                    model_name=HF_MODEL_NAME,
                    device=HF_DEVICE,
                    batch_size=32,
                    use_cache=True,
                    cache_dir='cache/hf_embeddings',
                )),
                ('clf', GaussianNB()),
            ]),
            'params': add_preproc_variants({
                'emb__normalize_embeddings': [True, False],
                'clf__var_smoothing': [1e-9, 1e-8, 1e-7],
            }),
        },
        {
            'name': 'E14_Preproc_HF_PCA_GaussianNB',
            'search_kind': 'random',
            'n_iter': HF_N_ITER,
            'pipeline': Pipeline([
                ('prep', TextPreprocessor()),
                ('emb', HFEmbeddingTransformer(
                    model_name=HF_MODEL_NAME,
                    device=HF_DEVICE,
                    batch_size=32,
                    use_cache=True,
                    cache_dir='cache/hf_embeddings',
                )),
                ('pca', PCA(random_state=SEED)),
                ('clf', GaussianNB()),
            ]),
            'params': add_preproc_variants({
                'emb__normalize_embeddings': [True, False],
                'pca__n_components': [64, 128],
                'clf__var_smoothing': [1e-9, 1e-8, 1e-7],
            }),
        },
    ]
    experiments.extend(hf_experiments)

if MAX_EXPERIMENTS > 0:
    experiments = experiments[:MAX_EXPERIMENTS]

print('Numero de experimentos:', len(experiments))
for e in experiments:
    print('-', e['name'])

Numero de experimentos: 14
- E01_Preproc_TFIDF_ComplementNB
- E02_Preproc_TFIDF_SVD_CatBoost
- E03_Preproc_TFIDF_SVD_LightGBM
- E04_Preproc_TFIDF_NMF_LightGBM
- E05_Preproc_Word2Vec_GaussianNB
- E06_Preproc_Word2Vec_PCA_GaussianNB
- E07_Preproc_FastText_GaussianNB
- E08_Preproc_FastText_PCA_GaussianNB
- E09_Preproc_Word2Vec_CatBoost
- E10_Preproc_Word2Vec_PCA_LightGBM
- E11_Preproc_FastText_LightGBM
- E12_Preproc_FastText_PCA_CatBoost
- E13_Preproc_HF_GaussianNB
- E14_Preproc_HF_PCA_GaussianNB


In [12]:
# Ejecucion de experimentos
results = []
search_map = {}

for exp in experiments:
    try:
        row, search = run_experiment(exp)
        results.append(row)
        search_map[exp['name']] = search
    except Exception as exc:
        print(f"Error en {exp['name']}: {exc}")
        results.append({
            'experiment': exp['name'],
            'status': 'error',
            'error': str(exc),
        })

results_df = pd.DataFrame(results)
results_df


===== E01_Preproc_TFIDF_ComplementNB =====
VAL f1_macro : 0.8411
TEST f1_macro: 0.8611
Candidatos   : 8
Tiempo (s)   : 73.88

===== E02_Preproc_TFIDF_SVD_CatBoost =====
VAL f1_macro : 0.8062
TEST f1_macro: 0.8002
Candidatos   : 8
Tiempo (s)   : 299.62

===== E03_Preproc_TFIDF_SVD_LightGBM =====
VAL f1_macro : 0.8111
TEST f1_macro: 0.8096
Candidatos   : 8
Tiempo (s)   : 308.89

===== E04_Preproc_TFIDF_NMF_LightGBM =====
VAL f1_macro : 0.8271
TEST f1_macro: 0.8178
Candidatos   : 8
Tiempo (s)   : 1288.14

===== E05_Preproc_Word2Vec_GaussianNB =====


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


VAL f1_macro : 0.7082
TEST f1_macro: 0.6975
Candidatos   : 8
Tiempo (s)   : 141.27

===== E06_Preproc_Word2Vec_PCA_GaussianNB =====


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

VAL f1_macro : 0.7694
TEST f1_macro: 0.7679
Candidatos   : 8
Tiempo (s)   : 151.62

===== E07_Preproc_FastText_GaussianNB =====


...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VAL f1_macro : 0.8176
TEST f1_macro: 0.8345
Candidatos   : 4
Tiempo (s)   : 57.96


,experiment,status,val_f1_macro,val_f1_weighted,val_accuracy,test_f1_macro,test_f1_weighted,test_accuracy,fit_time_sec,n_candidates,best_params
0,E01_Preproc_TFIDF_ComplementNB,ok,0.841114,0.866912,0.873016,0.861123,0.878051,0.882678,73.88,8,"{'tfidf__sublinear_tf': True, 'tfidf__ngram_ra..."
1,E02_Preproc_TFIDF_SVD_CatBoost,ok,0.806162,0.833919,0.835059,0.800249,0.834018,0.835749,299.62,8,"{'tfidf__ngram_range': (1, 1), 'tfidf__min_df'..."
2,E03_Preproc_TFIDF_SVD_LightGBM,ok,0.811072,0.846357,0.848861,0.809609,0.845513,0.846101,308.89,8,"{'tfidf__ngram_range': (1, 2), 'tfidf__min_df'..."
3,E04_Preproc_TFIDF_NMF_LightGBM,ok,0.827061,0.854437,0.856453,0.817785,0.845071,0.847481,1288.14,8,"{'tfidf__ngram_range': (1, 1), 'tfidf__min_df'..."
4,E05_Preproc_Word2Vec_GaussianNB,ok,0.708227,0.750447,0.746032,0.697542,0.754589,0.747412,141.27,8,"{'prep__stem': True, 'prep__remove_stopwords':..."
5,E06_Preproc_Word2Vec_PCA_GaussianNB,ok,0.769413,0.807643,0.803313,0.767919,0.810436,0.806073,151.62,8,"{'prep__stem': True, 'prep__remove_stopwords':..."
6,E07_Preproc_FastText_GaussianNB,ok,0.680252,0.729218,0.721877,0.665240,0.731503,0.721187,681.18,8,"{'prep__stem': True, 'prep__remove_stopwords':..."
7,E08_Preproc_FastText_PCA_GaussianNB,ok,0.753796,0.795110,0.790890,0.760158,0.800172,0.797792,801.59,8,"{'prep__stem': True, 'prep__remove_stopwords':..."
8,E09_Preproc_Word2Vec_CatBoost,ok,0.785216,0.819516,0.820566,0.779137,0.823042,0.824707,264.66,8,"{'prep__stem': True, 'prep__remove_stopwords':..."
9,E10_Preproc_Word2Vec_PCA_LightGBM,ok,0.774391,0.810299,0.812974,0.775057,0.817968,0.820566,347.79,8,"{'prep__stem': True, 'prep__remove_stopwords':..."


In [13]:
# Ranking comparativo y seleccion del ganador
ok_df = results_df[results_df['status'] == 'ok'].copy()

if ok_df.empty:
    raise RuntimeError('Ningun experimento finalizo correctamente.')

# Ganador definido SOLO por test_f1_macro
ok_df = ok_df.sort_values(
    by=['test_f1_macro'],
    ascending=False,
).reset_index(drop=True)

display_cols = [
    'experiment',
    'val_f1_macro',
    'val_f1_weighted',
    'val_accuracy',
    'test_f1_macro',
    'test_f1_weighted',
    'test_accuracy',
    'fit_time_sec',
    'n_candidates',
]

print('Ranking de experimentos (criterio: test_f1_macro):')
display(ok_df[display_cols])

winner_name = ok_df.iloc[0]['experiment']
print('Experimento ganador:', winner_name)

Ranking de experimentos (criterio: test_f1_macro):


,experiment,val_f1_macro,val_f1_weighted,val_accuracy,test_f1_macro,test_f1_weighted,test_accuracy,fit_time_sec,n_candidates
0,E01_Preproc_TFIDF_ComplementNB,0.841114,0.866912,0.873016,0.861123,0.878051,0.882678,73.88,8
1,E14_Preproc_HF_PCA_GaussianNB,0.817589,0.844908,0.843340,0.834470,0.860749,0.859903,57.96,4
2,E13_Preproc_HF_GaussianNB,0.807231,0.833935,0.832298,0.831967,0.857295,0.855072,92.92,4
3,E04_Preproc_TFIDF_NMF_LightGBM,0.827061,0.854437,0.856453,0.817785,0.845071,0.847481,1288.14,8
4,E03_Preproc_TFIDF_SVD_LightGBM,0.811072,0.846357,0.848861,0.809609,0.845513,0.846101,308.89,8
5,E02_Preproc_TFIDF_SVD_CatBoost,0.806162,0.833919,0.835059,0.800249,0.834018,0.835749,299.62,8
6,E09_Preproc_Word2Vec_CatBoost,0.785216,0.819516,0.820566,0.779137,0.823042,0.824707,264.66,8
7,E10_Preproc_Word2Vec_PCA_LightGBM,0.774391,0.810299,0.812974,0.775057,0.817968,0.820566,347.79,8
8,E06_Preproc_Word2Vec_PCA_GaussianNB,0.769413,0.807643,0.803313,0.767919,0.810436,0.806073,151.62,8
9,E08_Preproc_FastText_PCA_GaussianNB,0.753796,0.795110,0.790890,0.760158,0.800172,0.797792,801.59,8


Experimento ganador: E01_Preproc_TFIDF_ComplementNB


## Modelo ganador

In [14]:
# Evaluacion detallada del ganador
winner_result = search_map[winner_name]
winner_model = winner_result['best_model']

y_val_pred = winner_model.predict(X_val)
y_test_pred = winner_model.predict(X_test)

print('Best params del ganador:')
print(winner_result['best_params'])
print('Numero de candidatos evaluados:', winner_result['n_candidates'])

print('\nClassification report - VAL')
print(classification_report(y_val, y_val_pred, digits=4))

print('\nClassification report - TEST')
print(classification_report(y_test, y_test_pred, digits=4))

cm = confusion_matrix(y_test, y_test_pred)
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(cm).plot(ax=ax, colorbar=False)
ax.set_title(f'Matriz de confusion TEST - {winner_name}')
plt.tight_layout()
plt.show()

Best params del ganador:
{'tfidf__sublinear_tf': True, 'tfidf__ngram_range': (1, 2), 'tfidf__min_df': 2, 'prep__stem': False, 'prep__remove_stopwords': True, 'prep__lemmatize': True, 'clf__alpha': 0.3}
Numero de candidatos evaluados: 8

Classification report - VAL
              precision    recall  f1-score   support

           1     0.7632    0.7632    0.7632        76
           2     0.7895    0.8182    0.8036        55
           3     0.9343    0.9552    0.9446       134
           4     0.8580    0.9805    0.9152       154
           5     0.8848    0.9125    0.8985       160
           6     0.9252    0.9519    0.9384       104
           7     0.8286    0.9831    0.8992       118
           8     0.7778    0.4179    0.5437        67
           9     0.8500    0.6538    0.7391        52
          10     0.7750    0.5849    0.6667        53
          11     0.9167    0.8462    0.8800        91
          12     1.0000    0.7447    0.8537        47
          13     0.8400    0.900

In [15]:
# Evidencia de 4 predicciones de test
rng = np.random.default_rng(SEED)
idx = rng.choice(np.arange(len(X_test)), size=4, replace=False)

X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)
y_pred_reset = pd.Series(y_test_pred)

proba = None
if hasattr(winner_model, 'predict_proba'):
    try:
        proba = winner_model.predict_proba(X_test)
    except Exception:
        proba = None

rows = []
for i in idx:
    row = {
        'texto_preview': X_test_reset.iloc[i][:260].replace('\n', ' ') + ('...' if len(X_test_reset.iloc[i]) > 260 else ''),
        'ODS_real': int(y_test_reset.iloc[i]),
        'ODS_predicho': int(y_pred_reset.iloc[i]),
    }
    if proba is not None:
        row['confianza_max'] = float(np.max(proba[i]))
    rows.append(row)

pd.DataFrame(rows)

,texto_preview,ODS_real,ODS_predicho,confianza_max
0,La vivienda también puede resultar inasequible...,11,1,0.099168
1,"Sin embargo, aunque indica que el agua no esca...",6,6,0.308725
2,El aumento de la movilidad internacional de lo...,3,3,0.173811
3,En los cinco países se han tomado medidas para...,5,5,0.156929


In [16]:
# Exportables
artifacts_dir = Path('artifacts')
artifacts_dir.mkdir(parents=True, exist_ok=True)

ok_df.to_csv(artifacts_dir / 'experiment_results.csv', index=False)

try:
    joblib.dump(winner_model, artifacts_dir / 'best_model.joblib')
    model_saved = True
except Exception as exc:
    model_saved = False
    print('No se pudo serializar el modelo ganador con joblib:', exc)

metadata = {
    'winner_experiment': winner_name,
    'best_params': winner_result['best_params'],
    'n_candidates': winner_result['n_candidates'],
    'val_f1_macro': float(f1_score(y_val, y_val_pred, average='macro')),
    'test_f1_macro': float(f1_score(y_test, y_test_pred, average='macro')),
    'seed': SEED,
    'train_size': int(len(X_train)),
    'val_size': int(len(X_val)),
    'test_size': int(len(X_test)),
    'hf_experiments_enabled': ENABLE_HF_EXPERIMENTS,
    'hf_device': HF_DEVICE,
    'model_saved_joblib': model_saved,
}

(artifacts_dir / 'best_model_metadata.json').write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('Exportables generados en:', artifacts_dir.resolve())
print('best_model.joblib guardado:', model_saved)

Exportables generados en: /Users/adrianalarcon/Library/CloudStorage/OneDrive-UniversidaddelosAndes/MAIA/aprendizaje_no_supervisado/assignment_2/artifacts
best_model.joblib guardado: True


## Notas operativas

- Para corrida completa: dejar variables por defecto.
- Para smoke test rapido:
  - `USE_DEV_SAMPLE=1`
  - `FAST_MODE=1`
  - `MAX_EXPERIMENTS=3`
  - `ENABLE_HF_EXPERIMENTS=0` (si no hay MPS)
  - `LIMIT_PREPROC_VARIANTS=1`

Optimizacion de embeddings avanzados:
- solo 2 experimentos HF (`E13` sin PCA y `E14` con PCA),
- un unico modelo HF (`HF_MODEL_NAME`),
- cache activo en `cache/hf_embeddings`,
- `HF_N_ITER` bajo para reducir tiempo.

Nota: este notebook no usa CV; el tuning se hace solo con `train` y seleccion por `val`.